<a href="https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/fix/madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLI CityLearn v3 MADRL

Este notebook sigue la estructura del ejemplo `cli.ipynb` original de CityLearn, adaptado al proyecto **CityLearn v3 MADRL**: 17 edificios + EV, tres ejes `E1/E2/E3`, recompensa `CityLearnV3MADRLRewardFunction` y backends HAPPO, MASAC, MATD3 y MAAC.

En CityLearn v2 el CLI principal se invoca con `python -m citylearn`; se conserva para datasets, variables y simulaciones compatibles. El entrenamiento MADRL tiene **una sola via de lanzamiento**: la CLI de la campaña nueva

```
python -m uc3m.application.training.cli --config configs/experiments/iquitos.yaml --stage prepare
python -m uc3m.application.training.cli --config configs/experiments/iquitos.yaml --stage {pilot,hpo,confirm,sensitivity} [--execute]
```

Sin `--execute` nada se entrena. Con `--execute` solo corren las etapas autorizadas en la configuracion, y cada intento empieza desde cero. Guia operativa: `docs/architecture/runbook.md`.

## Install or Open the Project

En una maquina local del proyecto no es necesario reinstalar. En Google Colab se clona el repositorio padre con sus submodulos fijados y se instala CityLearn en modo editable; la instalacion Python 3.9 completa esta en la celda **1.3** de `madrl_citylearn_v3_tutorial.ipynb`. La celda queda apagada por defecto para no modificar el entorno accidentalmente.

In [ ]:
# Controla si se clona e instala el proyecto en Google Colab.
RUN_COLAB_INSTALL = False

# Importa utilidades del sistema operativo.
import os

# Importa rutas portables para Windows, Linux y Colab.
from pathlib import Path

# Ejecuta comandos externos cuando una celda lo requiere.
import subprocess

# Expone el interprete Python activo.
import sys

# Clona el repositorio padre con sus submodulos fijados solo si se activa la bandera.
if RUN_COLAB_INSTALL:
    # Clona el proyecto completo: CityLearn y external/ quedan en los commits fijados por el padre.
    subprocess.run(['git', 'clone', '--recursive', 'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git'], check=True)
    # Cambia el directorio activo a la raiz del proyecto clonado.
    os.chdir('MADRLCitytleranflexresdr')
    # Instala CityLearn en modo editable (la instalacion completa Python 3.9 esta en la celda 1.3 del tutorial).
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', 'CityLearn/'], check=True)
# Mantiene la celda segura cuando no se instala nada.
else:
    # Informa que no se modifico el entorno.
    print('RUN_COLAB_INSTALL=False; usando el entorno actual.')

## Project Paths

La celda detecta la raiz del proyecto padre (carpeta con `uc3m/` y `configs/experiments/`) aunque el notebook se ejecute desde `examples_madrl_v3/`. Define la configuracion de campaña, el dataset congelado `data/campaigns/iquitos_v6_trial13/` y la raiz de artefactos `outputs/campaigns/iquitos_v6_trial13/`.

In [ ]:
# Importa JSON para leer la salida de la CLI de campaña.
import json

# Detecta la raiz del proyecto padre: contiene uc3m/ y configs/experiments/.
def find_project_root(start=None):
    # Parte del directorio actual del notebook.
    start = Path(start or Path.cwd()).resolve()
    # Recorre el directorio actual, sus padres y la ruta estandar de Colab.
    for candidate in (start, *start.parents, Path('/content/MADRLCitytleranflexresdr')):
        # Acepta la primera carpeta con la estructura del proyecto.
        if (candidate / 'uc3m').is_dir() and (candidate / 'configs' / 'experiments').is_dir():
            return candidate
    # Falla de forma visible si el notebook no esta dentro del proyecto.
    raise FileNotFoundError('No se encontro la raiz del proyecto (uc3m/ + configs/experiments/).')

# Fija la raiz del proyecto padre.
PROJECT_ROOT = find_project_root()

# Fija la raiz del simulador CityLearn (dependencia protegida, solo lectura).
CITYLEARN_ROOT = PROJECT_ROOT / 'CityLearn'

# Agrega proyecto y CityLearn al path de Python del kernel.
for _path in (str(CITYLEARN_ROOT), str(PROJECT_ROOT)):
    if _path not in sys.path:
        sys.path.insert(0, _path)

# Usa el Python 3.9 del proyecto si una celda previa lo definio.
PYTHON = globals().get('PROJECT_PYTHON', sys.executable)

# Configuracion unica de la campaña nueva.
CAMPAIGN_CONFIG = 'configs/experiments/iquitos.yaml'

# Dataset congelado de la campaña (splits train_2023 / validation_2024 / test_2025).
DATASET_ROOT = PROJECT_ROOT / 'data/campaigns/iquitos_v6_trial13'

# Raiz de artefactos de la campaña nueva; no existe hasta ejecutar --stage prepare.
ARTIFACT_ROOT = PROJECT_ROOT / 'outputs/campaigns/iquitos_v6_trial13'

# Manifiesto de la campaña escrito por la etapa prepare.
MANIFEST_PATH = ARTIFACT_ROOT / 'campaign_manifest.json'

# Entorno de los subprocesos: la CLI importa uc3m y CityLearn desde el arbol local.
def cli_env():
    env = dict(os.environ)
    env['PYTHONPATH'] = os.pathsep.join([str(PROJECT_ROOT), str(CITYLEARN_ROOT), env.get('PYTHONPATH', '')])
    return env

# Muestra el codigo de salida y el JSON que imprime la CLI de campaña.
def show_cli_result(result):
    print('exit code =', result.returncode)
    lines = [line for line in (result.stdout or '').splitlines() if line.strip()]
    if lines:
        try:
            print(json.dumps(json.loads(lines[-1]), indent=2))
        except ValueError:
            print('\n'.join(lines[-20:]))
    if result.returncode != 0 and result.stderr:
        print(result.stderr[-3000:])

# Muestra las rutas clave para diagnostico.
print('PROJECT_ROOT  =', PROJECT_ROOT)
print('CAMPAIGN      =', PROJECT_ROOT / CAMPAIGN_CONFIG)
print('DATASET_ROOT  =', DATASET_ROOT, '(existe)' if DATASET_ROOT.is_dir() else '(FALTA)')
print('ARTIFACT_ROOT =', ARTIFACT_ROOT, '(preparada)' if MANIFEST_PATH.is_file() else '(aun no preparada)')

# Define los algoritmos y ejes de la campaña (fuente: configs/experiments/iquitos.yaml).
algorithms = ['HAPPO', 'MASAC', 'MATD3', 'MAAC']

# Define los tres ejes del proyecto.
scenarios = ['E1', 'E2', 'E3']

## CityLearn Base CLI Documentation

El CLI base de CityLearn v2 sigue disponible. Sirve para listar datasets, variables por defecto y ejecutar agentes compatibles con `citylearn.agents` o `stable-baselines3`.

In [ ]:
# Construye el comando de ayuda del CLI base.
citylearn_help_command = [PYTHON, '-m', 'citylearn', '-h']

# Muestra el comando que se ejecutara.
print('Running:', ' '.join(citylearn_help_command))

# Ejecuta la ayuda del CLI base de CityLearn.
subprocess.run(citylearn_help_command, check=False, cwd=CITYLEARN_ROOT)

## Running a Simulation using the Original CLI

La suborden `simulate` es parte de CityLearn v2 y se conserva para agentes base, RBC y evaluaciones compatibles. Los cuatro MADRL del proyecto no se lanzan desde aqui: se entrenan solo a traves de la CLI de campaña, que construye cada intento con su dataset congelado, sus pesos de escenario y su evidencia de aprendizaje.

In [ ]:
# Construye el comando de ayuda de la suborden simulate.
simulate_help_command = [PYTHON, '-m', 'citylearn', 'simulate', '-h']

# Muestra el comando que se ejecutara.
print('Running:', ' '.join(simulate_help_command))

# Ejecuta la ayuda de simulate.
subprocess.run(simulate_help_command, check=False, cwd=CITYLEARN_ROOT)

## CityLearn v3 MADRL Campaign CLI

La CLI `uc3m.application.training.cli` es el unico punto de entrada de entrenamiento. Cada algoritmo conserva su backend nativo (HAPPO on-policy HARL, MASAC con estado global CTDE, MATD3 off-policy continuo, MAAC con critico de atencion), pero la CLI es quien arma sus argumentos a partir de `configs/algorithms/native.yaml`.

| Etapa | Que hace |
|---|---|
| `prepare` | Construye el DAG de tratamientos y el presupuesto; no entrena |
| `pilot` | 4 algoritmos, E1, semilla 901, 2 episodios de 168 pasos: mide recursos |
| `hpo` | Rejilla de hiperparametros con semillas de desarrollo 901-903; seleccion en `validation_2024` |
| `confirm` | Configuracion congelada, 4 algoritmos x 3 escenarios x 12 semillas 1001-1012, 50 x 8760 pasos |
| `sensitivity` | Dominancia p=0.55/0.65 del eje principal sobre las semillas 1001-1003 |

La ayuda `-h` muestra los parametros sin iniciar entrenamiento.

In [ ]:
# Construye el comando de ayuda de la CLI de campaña.
campaign_help_command = [PYTHON, '-B', '-m', 'uc3m.application.training.cli', '-h']

# Muestra el comando exacto.
print(' '.join(map(str, campaign_help_command)))

# Ejecuta la ayuda sin iniciar entrenamiento.
subprocess.run(campaign_help_command, check=False, cwd=PROJECT_ROOT, env=cli_env())

## Prepare the Campaign (no training)

La etapa `prepare` lee `configs/experiments/iquitos.yaml`, verifica el dataset congelado y escribe `outputs/campaigns/iquitos_v6_trial13/campaign_manifest.json` mas `stages/{pilot,hpo,confirm,sensitivity}.json` con el presupuesto de pasos. No entrena ni consulta la GPU. Si la campaña ya esta preparada, la CLI comprueba que sus entradas no cambiaron; un cambio exige una identidad de campaña nueva.

In [ ]:
# Prepara la campaña: escribe manifiestos y presupuesto. NO entrena ni consulta la GPU.
prepare_command = [PYTHON, '-B', '-m', 'uc3m.application.training.cli',
                   '--config', 'configs/experiments/iquitos.yaml', '--stage', 'prepare']

# Muestra el comando exacto.
print(' '.join(map(str, prepare_command)))

# Ejecuta la preparacion; si la campaña ya existe, la CLI solo verifica que sus entradas no cambiaron.
prepare_result = subprocess.run(prepare_command, cwd=PROJECT_ROOT, env=cli_env(),
                                capture_output=True, text=True, check=False)

# Muestra el JSON de la CLI (status prepared_not_executed y presupuesto de pasos).
show_cli_result(prepare_result)

## Run a Stage (explicit opt-in)

La misma CLI ejecuta las etapas en Windows, Linux o Colab; no hay comandos por algoritmo ni lanzadores por plataforma. En Colab se usa el cuaderno `madrl_citylearn_v3_modular_colab.ipynb`, que llama a esta CLI y copia cada intento a Drive.

`EXECUTE_STAGE = False` mantiene la celda segura: la CLI solo describe la etapa. Para entrenar, el propietario autoriza la etapa en `resource_authorization.authorized_stages`, se prepara de nuevo y se cambia la bandera a `True`. Una interrupcion deja el intento como evidencia fallida; relanzar la etapa crea un intento nuevo desde el episodio 0.

In [ ]:
# Etapa a consultar o ejecutar: 'pilot', 'hpo', 'confirm' o 'sensitivity'.
RUN_STAGE = 'pilot'

# Opt-in explicito. Con False la CLI solo informa el plan de la etapa y no entrena.
# Con True solo corre si la etapa figura en resource_authorization.authorized_stages
# de configs/experiments/iquitos.yaml; cada intento empieza desde cero (sin reanudacion).
EXECUTE_STAGE = False

# Construye el comando de la unica via de lanzamiento.
stage_command = [PYTHON, '-B', '-m', 'uc3m.application.training.cli',
                 '--config', 'configs/experiments/iquitos.yaml', '--stage', RUN_STAGE]
stage_command += ['--execute'] if EXECUTE_STAGE else []

# Muestra el comando exacto.
print(' '.join(map(str, stage_command)))

# Ejecuta la CLI: sin --execute imprime prepared_not_executed con el plan de la etapa.
stage_result = subprocess.run(stage_command, cwd=PROJECT_ROOT, env=cli_env(),
                              capture_output=True, text=True, check=False)

# Muestra el JSON de la CLI (not_authorized, training_complete o incomplete).
show_cli_result(stage_result)

## Campaign Status

El estado se lee de los manifiestos de la campaña nueva: `campaign_manifest.json` y `stages/<stage>_result.json`. Cada intento vive en `outputs/campaigns/iquitos_v6_trial13/<stage>/<algoritmo>/<escenario>/<candidato>/seed_<semilla>/try_<n>/`. Si la campaña no esta preparada o una etapa no se ejecuto, la celda lo dice; no muestra numeros de corridas anteriores.

In [ ]:
# Lee el estado real de la campaña desde sus manifiestos; nunca desde corridas anteriores.
if not MANIFEST_PATH.is_file():
    # Sin manifiesto no hay campaña preparada ni entrenamiento que reportar.
    print('Campaña no preparada: ejecuta la etapa prepare. No hay resultados de entrenamiento.')
else:
    # Carga el manifiesto de la campaña.
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    # Muestra identidad y estado declarados.
    print('campaign_id =', manifest['campaign_id'], '| status =', manifest['status'])
    # Recorre las etapas declaradas en el manifiesto.
    for stage, record in manifest['stages'].items():
        # Cada etapa ejecutada escribe stages/<stage>_result.json.
        result_path = ARTIFACT_ROOT / 'stages' / f'{stage}_result.json'
        if not result_path.is_file():
            print(f"{stage:12s} {record['treatments']:4d} tratamientos planificados | sin ejecutar")
            continue
        result = json.loads(result_path.read_text(encoding='utf-8'))
        print(f"{stage:12s} {result['completed_treatments']}/{result['planned_treatments']} completos | "
              f"intentos fallidos={result['failed_attempts']} | status={result['status']}")

## Outputs and Validation

Cada intento deja `attempt.json`, `child.log`, `learning_evidence.json`, `supervision.json` y `completion.json`. Un intento solo cuenta si el optimizador aplico actualizaciones reales, la trayectoria alcanzo episodios x horas declarados, no hubo *salvage* y el checkpoint es recargable. La puerta `verify_fresh_acceptance` separa aceptacion de software y aceptacion cientifica; esta ultima requiere evaluacion congelada y reporte estadistico completos.

In [ ]:
# Verifica la aceptacion de la campaña nueva con la puerta oficial (solo lectura).
from uc3m.application.training.fresh_acceptance import verify_fresh_acceptance

if not MANIFEST_PATH.is_file():
    # Sin campaña preparada no hay nada que aceptar.
    print('Campaña no preparada: no hay modelos entrenados ni aceptacion que verificar.')
else:
    # La puerta revisa identidad, hashes, intentos completos, checkpoints y reporte cientifico.
    report = verify_fresh_acceptance(MANIFEST_PATH)
    print(json.dumps(report.as_dict(), indent=2))
    # not_ready_no_model significa que todavia no hay ningun modelo entrenado y verificado.
    if report.status == 'not_ready_no_model':
        print('Todavia no hay modelos entrenados y verificados en esta campaña.')